In [1]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ
using Condor

In [2]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(10.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;

In [3]:
cs = ConvolutionSky(lmax = lmax_in, numofsky = 1)
@show fieldnames(typeof(cs))
cb = ConvolutionBeam(lmax = lmax_in, mmax =2, numofbeams = 1)
@show fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
@show fieldnames(typeof(cc))

fieldnames(typeof(cs)) = (:numofsky, :lmax, :alm)
fieldnames(typeof(cb)) = (:numofbeams, :lmax, :mmax, :blm)
fieldnames(typeof(cc)) = (:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)


(:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)

In [4]:
#cb.mmax = 2
cb.blm[:,:,1] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
#cb.blm[:,:,2] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
;

In [5]:
alm_in = npzread("./inputs/alm=CMB=r0=nside32=seed200.npy")
alm_in[1,:] .= 0
cs.alm[1,:,:] = alm_in
#cs.alm[2,:,:] = alm_in*0.9
#cs.alm[3,:,:] = alm_in*0.8
#cs.alm[4,:,:] = alm_in*0.7
#cs.alm[5,:,:] = alm_in*0.6

alm_smooth = npzread("./inputs/alm_smooth=CMB=r0=nside32=seed200.npy")
alm_smooth[1,:] .= 0
alm2 = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, Healpix.numberOfAlms(cs.lmax, cs.lmax)))
alm2.alm = alm_smooth[1,:]
mT = Healpix.alm2map(alm2, nside_in)   # 戻りは Healpix.Map

nalm  = Healpix.numberOfAlms(cs.lmax, cs.lmax)
almT = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almE = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almB = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almT.alm = alm_smooth[1,:]
almE.alm = alm_smooth[2,:]
almB.alm = alm_smooth[3,:]

maps= alm2map([almT, almE, almB], nside_in)

PolarizedHealpixMap{Float64, RingOrder, Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [-0.17789798126166229, 0.18094867555043045, -0.17205116330882814, 0.1644002988002581, -0.16609571338993107, -0.06904765307853662, 0.1947429989451293, 0.059594696479038506, -0.19493511695283766, -0.050323476795787275  …  -0.19920704181240317, -0.10239775255592834, 0.2175308008112957, 0.11396908487886111, -0.21570285448853949, -0.11051959918221216, 0.22106163692784328, -0.2213245287671585, 0.2323454252407058, -0.22863192233663263], [0.08288250992957649, -0.09593691661051275, 0.09429627834347007, -0.07954684465208599, -0.07677563580779123, 0.18549613585585842, 0.06173371538532244, -0.19705322476125087, -0.05632741125917598, 0.1794914523399764  …  0.09680969961502273, -0.20552522634604112, -0.11157295995121179, 0.2222791556343957, 0.11259726561071787, -0.20937268077287927, 0.07749039012325322, -0.07592412263070766, 0.0773306278685

In [28]:
alm_slice = slice_spin_alm_by_l(cs, cc)
blm_slice = slice_spin_blm_by_l(cb, cc);

In [29]:
pix_idx = 60
θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)

(0.12766426866687355, 6.126105674500097)

In [45]:
nptg = 100
θ = θ_pix
φ = φ_pix
dθ = rand(nptg)*1e-5
dφ = rand(nptg)*1e-5*0
ψ = rand(nptg)
globalD = global_wignerD_conj(cc, φ, θ, 0)

;

In [83]:
@time test = compute_pixel_convolution(pix_idx, globalD, φ.+dφ, θ.+dθ, ψ; τ=10)

  1.073440 seconds (761.83 k allocations: 57.700 MiB, 0.62% gc time, 30.95% compilation time)


1×1×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -0.05996602000713844 + 1.8041124150158794e-16im

[:, :, 2] =
 -0.010620448812164365 - 0.05629629956184677im

[:, :, 3] =
 -0.010620448812164136 + 0.056296299561847im

In [47]:
function mapmake(convolved_date, ψ)
    result =  similar(convolved_date, Float64) 
    h2 = mean(exp.(2im*ψ))
    h4 = mean(exp.(4im*ψ))
    M = [1.0 conj(h2)/2 h2/2;
        h2/2 1/4 (h4)/4;
        conj(h2)/2 conj(h4)/4 1/4]
    for i in 1:cs.numofsky
        for j in 1:cb.numofbeams
            d_array = [convolved_date[i,j,1]; convolved_date[i,j,2]/2; convolved_date[i,j,3]/2]
            temp = inv(M)* d_array
            result[i,j,1] = real(temp[1])
            result[i,j,2] = real(temp[2])
            result[i,j,3] = imag(temp[2])
        end
    end
    return result
end

mapmake (generic function with 1 method)

In [48]:
@show i = maps.i[pix_idx]
@show p = maps.q[pix_idx]+im*maps.u[pix_idx]
pp = conj(p)
@show mean(i .+ 1/2 .* p .* exp.(-2im*ψ[:]) .+ 1/2 .* pp .*exp.(2im*ψ[:]))
@show mean(i.*exp.(2im*ψ[:]) .+ 1/2 .* p  .+ 1/2 .* pp .*exp.(4im*ψ[:]))
@show mean(i.*exp.(-2im*ψ[:]) .+ 1/2 .* p.*exp.(-4im*ψ[:])  .+ 1/2 .* pp)


i = maps.i[pix_idx] = 0.0
p = maps.q[pix_idx] + im * maps.u[pix_idx] = 0.036397435260783255 - 0.10749622730359497im
mean((i .+ ((1 / 2) .* p) .* exp.((-2im) * ψ[:])) .+ ((1 / 2) .* pp) .* exp.((2im) * ψ[:])) = -0.061301968729653246 + 0.0im
mean((i .* exp.((2im) * ψ[:]) .+ (1 / 2) .* p) .+ ((1 / 2) .* pp) .* exp.((4im) * ψ[:])) = -0.010857930895929563 - 0.05754989684492278im
mean((i .* exp.((-2im) * ψ[:]) .+ ((1 / 2) .* p) .* exp.((-4im) * ψ[:])) .+ (1 / 2) .* pp) = -0.010857930895929563 + 0.05754989684492278im


-0.010857930895929563 + 0.05754989684492278im

In [49]:
h2 = mean(exp.(2im*ψ))
h4 = mean(exp.(4im*ψ))
M = [1.0 conj(h2)/2 h2/2;
        h2/2 1/4 (h4)/4;
        conj(h2)/2 conj(h4)/4 1/4]
        
d_1 =  [mean(i .+ 1/2 .* p .* exp.(-2im*ψ[:]) .+ 1/2 .* pp .*exp.(2im*ψ[:]));
mean(i.*exp.(2im*ψ[:]) .+ 1/2 .* p  .+ 1/2 .* pp .*exp.(4im*ψ[:]))/2;
mean(i.*exp.(-2im*ψ[:]) .+ 1/2 .* p.*exp.(-4im*ψ[:])  .+ 1/2 .* pp )/2
]

d_2 =  [test[1,1,1];
(test[1,1,2])/2;
(test[1,1,3])/2
]

3-element Vector{ComplexF64}:
  -0.05996602000713844 + 1.8041124150158794e-16im
 -0.005310224406082182 - 0.028148149780923386im
 -0.005310224406082068 + 0.0281481497809235im

In [50]:
d_1

3-element Vector{ComplexF64}:
 -0.061301968729653246 + 0.0im
 -0.005428965447964782 - 0.02877494842246139im
 -0.005428965447964782 + 0.02877494842246139im

In [51]:
real.(d_2)./real.(d_1)

3-element Vector{Float64}:
 0.9782070829012612
 0.978128237687143
 0.9781282376871219

In [52]:
inv(M)*d_1
#M \ d_1

3-element Vector{ComplexF64}:
 5.635182063108523e-16 + 4.23620421201536e-18im
   0.03639743526078303 - 0.10749622730359551im
  0.036397435260783075 + 0.10749622730359579im

In [53]:
inv(M)*d_2

3-element Vector{ComplexF64}:
 7.386509628298253e-7 + 1.0368013155634005e-17im
  0.03560737151143862 - 0.10515658077186235im
 0.035607371511438855 + 0.10515658077186305im

In [54]:
p

0.036397435260783255 - 0.10749622730359497im

In [55]:
test_map = compute_pixel_convolution_mapmake(cs,cb,cc,pix_idx, globalD, φ.+dφ, θ.+dθ, ψ; τ=10)

1×1×3 Array{Float64, 3}:
[:, :, 1] =
 7.386509628298253e-7

[:, :, 2] =
 0.03560737151143862

[:, :, 3] =
 -0.10515658077186235

In [69]:
using Condor

In [58]:
nside = 32
lmax = 3*nside-1
fwhm = deg2rad(10)
kmax = 2

2

In [70]:
#blm = hp.blm_gauss(fwhm = fwhm, lmax = lmax, pol = true)
blm_len = alm_idx(l=lmax, m=lmax, lmax=lmax, mmax= lmax)
blm_calc = zeros(ComplexF64, 3, blm_len)
idx = alm_idx(l=lmax, m=kmax, lmax=lmax, mmax= 2)
blm_calc[1,1:idx] .= blm_harmonic.blmT.alm
blm_calc[2,1:idx] .= blm_harmonic.blmE.alm
blm_calc[3,1:idx] .= blm_harmonic.blmB.alm;

In [72]:
cp_ = Condor.gen_ConvolutionParams()
cp_.npix = nside2npix(nside) # Int npix
cp_.lmax = lmax # Int lmax
cp_.alm = alm_in[1:3,:]
cp_.blm = blm_calc
cp_.l_range = [0,lmax] # Int ell's calculate range
;

In [76]:
mueller = [1 0 0 0;
            0 1 0 0;
            0 0 1 0;
            0 0 0 1]

4×4 Matrix{Int64}:
 1  0  0  0
 0  1  0  0
 0  0  1  0
 0  0  0  1

In [77]:
todbc = @time Condor.tod_convolution_like_bc_new(cp_, θ.+dθ, φ.+dφ, ψ,  ψ*0 , mueller);

  4.931041 seconds (3.82 M allocations: 1.444 GiB, 2.00% gc time, 24.98% compilation time)


In [92]:
@show mean(todbc)
@show mean(todbc.*exp.(2im.*ψ))
@show mean(todbc.*exp.(-2im.*ψ))

@show test[:,:,1]/mean(todbc)
@show test[:,:,2]/mean(todbc.*exp.(2im.*ψ))
@show test[:,:,3]/mean(todbc.*exp.(-2im.*ψ))

mean(todbc) = -0.05996602002232932
mean(todbc .* exp.((2im) .* ψ)) = -0.010620448818220652 - 0.05629629957386418im
mean(todbc .* exp.((-2im) .* ψ)) = -0.010620448818220652 + 0.05629629957386418im
test[:, :, 1] / mean(todbc) = ComplexF64[0.9999999997466752 - 3.0085578705141493e-15im;;]
test[:, :, 2] / mean(todbc .* exp.((2im) .* ψ)) = ComplexF64[0.9999999997742715 + 6.499448425643825e-11im;;]
test[:, :, 3] / mean(todbc .* exp.((-2im) .* ψ)) = ComplexF64[0.9999999997742747 - 6.499915582016839e-11im;;]


1×1 Matrix{ComplexF64}:
 0.9999999997742747 - 6.499915582016839e-11im

In [88]:
@show mean(i .+ 1/2 .* p .* exp.(-2im*ψ[:]) .+ 1/2 .* pp .*exp.(2im*ψ[:]))
@show mean(i.*exp.(2im*ψ[:]) .+ 1/2 .* p  .+ 1/2 .* pp .*exp.(4im*ψ[:]))
@show mean(i.*exp.(-2im*ψ[:]) .+ 1/2 .* p.*exp.(-4im*ψ[:])  .+ 1/2 .* pp)

mean((i .+ ((1 / 2) .* p) .* exp.((-2im) * ψ[:])) .+ ((1 / 2) .* pp) .* exp.((2im) * ψ[:])) = -0.061301968729653246 + 0.0im
mean((i .* exp.((2im) * ψ[:]) .+ (1 / 2) .* p) .+ ((1 / 2) .* pp) .* exp.((4im) * ψ[:])) = -0.010857930895929563 - 0.05754989684492278im
mean((i .* exp.((-2im) * ψ[:]) .+ ((1 / 2) .* p) .* exp.((-4im) * ψ[:])) .+ (1 / 2) .* pp) = -0.010857930895929563 + 0.05754989684492278im


-0.010857930895929563 + 0.05754989684492278im

In [ ]:
blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm

In [64]:
function alm_idx(; l::Integer, m::Integer, lmax::Integer, mmax::Integer=lmax)
    (0 ≤ m ≤ mmax) || throw(ArgumentError("m must be in [0, mmax]"))
    (m ≤ l ≤ lmax) || throw(ArgumentError("l must satisfy m ≤ l ≤ lmax"))

    # offset for all previous m-blocks
    offset = m * (lmax + 1) - (m * (m - 1)) ÷ 2
    return Int(offset + (l - m) + 1)  # 1-based
end


alm_idx (generic function with 1 method)

In [67]:
alm_idx(l=lmax, m=2, lmax=lmax, mmax= 2)

285

In [150]:
function compute_pixel_convolution_mapmake(cs,cb,cc,pix_idx, globalD, φ, θ, ψ; τ=5)
    α_local, β_local, γ_local = calc_local_euiler_angles(cc.resol, pix_idx, φ, θ, ψ)
    localD = local_effective_wignerD_conj_reduced_formapmake(cb, cc, α_local, β_local, γ_local, ψ, τ=τ)
    convolved_date = convolver_1pixel(cs, cb, cc, alm_slice, blm_slice, globalD, localD)
    result = mapmake(convolved_date, ψ)
    return result
end

compute_pixel_convolution_mapmake (generic function with 1 method)

mapmake (generic function with 1 method)

In [56]:
mapmake(test, ψ)

In [58]:
test

5×2×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -31.1907+9.71851e-13im  -31.1907+9.71851e-13im
 -28.0716+8.70404e-13im  -28.0716+8.70404e-13im
 -24.9525+7.841e-13im    -24.9525+7.841e-13im
 -21.8335+6.7917e-13im   -21.8335+6.7917e-13im
 -18.7144+5.86175e-13im  -18.7144+5.86175e-13im

[:, :, 2] =
 3.26102-31.0203im  3.26102-31.0203im
 2.93492-27.9182im  2.93492-27.9182im
 2.60882-24.8162im  2.60882-24.8162im
 2.28272-21.7142im  2.28272-21.7142im
 1.95661-18.6122im  1.95661-18.6122im

[:, :, 3] =
 3.26102+31.0203im  3.26102+31.0203im
 2.93492+27.9182im  2.93492+27.9182im
 2.60882+24.8162im  2.60882+24.8162im
 2.28272+21.7142im  2.28272+21.7142im
 1.95661+18.6122im  1.95661+18.6122im

In [51]:
test[1,1,:]*0.9

3-element Vector{ComplexF64}:
 -28.076871456539305 + 8.753966301844329e-13im
  -2.845959420964878 - 27.927896784545144im
  -2.845959420964878 + 27.927896784545144im

In [50]:
test[2,2,:]

3-element Vector{ComplexF64}:
 -28.076871456539358 + 8.711052712495615e-13im
 -2.8459594209648835 - 27.927896784545165im
 -2.8459594209648835 + 27.927896784545165im

In [183]:
α_local, β_local, γ_local = calc_local_euiler_angles(res, pix_idx, φ.+dφ, θ.+dθ, ψ)
;

In [186]:
γ_local

10-element Vector{Float64}:
  0.03879249459094458
 -2.7416072511905694
  3.1003900264123407
  1.0407820765384024
  1.7288107522602552
  2.777370172006682
  1.3706202436165125
 -2.2189597965797487
  2.2843840646688074
  1.6346265790892058

In [35]:
lD = local_effective_wignerD_conj_reduced_formapmake(cb, cc, α_local, β_local, γ_local, ψ, τ=5)
globalD = global_wignerD_conj(cc, φ, θ, 0)
;

In [17]:
test = @time convolver_1pixel(cs, cb, cc, alm_slice, blm_slice, gD, lD)

  0.139102 seconds (91 allocations: 5.197 MiB, 58.99% gc time)


1×2×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -31.1885+9.64982e-13im  -31.1885+9.64982e-13im

[:, :, 2] =
 2.0477+9.84558im  2.0477+9.84558im

[:, :, 3] =
 2.0477-9.84558im  2.0477-9.84558im

In [15]:
function compute_pixel_convolution(pix_idx, globalD, φ, θ, ψ; τ=5)
    α_local, β_local, γ_local = calc_local_euiler_angles(cc.resol, pix_idx, φ, θ, ψ)
    localD = local_effective_wignerD_conj_reduced_formapmake(cb, cc, α_local, β_local, γ_local, ψ, τ=τ)
    result = convolver_1pixel(cs, cb, cc, alm_slice, blm_slice, globalD, localD)
    return result
end

compute_pixel_convolution (generic function with 1 method)

In [86]:
@time test = compute_pixel_convolution(pix_idx, globalD, φ, θ, ψ; τ=5)

  0.032659 seconds (132 allocations: 3.255 MiB)


1×1×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -31.21967242222391 + 9.712757128525068e-13im

[:, :, 2] =
 -31.125648936489714 - 2.420034743478086im

[:, :, 3] =
 -31.125648936489714 + 2.420034743478086im

In [223]:
alm_smooth = npzread("alm_smooth=CMB=r0=nside32=seed1.npy")

alm2 = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, Healpix.numberOfAlms(cs.lmax, cs.lmax)))
alm2.alm = alm_smooth[1,:]
mT = Healpix.alm2map(alm2, nside_in)   # 戻りは Healpix.Map

nalm  = Healpix.numberOfAlms(cs.lmax, cs.lmax)
almT = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almE = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almB = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almT.alm = alm_smooth[1,:]*0
almE.alm = alm_smooth[2,:]
almB.alm = alm_smooth[3,:]

maps= alm2map([almT, almE, almB], nside_in)

PolarizedHealpixMap{Float64, RingOrder, Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.3841684359262724, -0.2283283590288716, -0.588348244867616, -0.8973271051817646, -0.0369796019997603, 0.053658932027735334, -0.04304553931855282, -0.028340964839986604, -0.5793540400447466, 0.11914016822512552  …  0.034733190068783434, -0.4229562186824516, -0.01814141514429357, 0.036842700099326456, -0.20377726778227395, -0.06944079633454092, 0.4894117687322749, -0.43221459323236966, 0.013511342306182589, -0.2992332489751508], [-0.12281782625132895, 0.2541801993465586, 0.3691346888610124, -0.29191966435191946, 0.22773662495005553, 0.3644901510742654, -0.49185967415295534, -0.22279154538813686, 0.01558872821688971, 0.049620991313701485  …  0.6559649745223178, -0.14795975111327847, -0.444883355427664, 0.19488222021154258, 0.052515494108757, -0.363782508427228, -0.26587105624820717, 0.3398128452584115, -0.255309736954928, -0.17

In [106]:
maps.i[pix_idx]

-31.1911553811341

In [107]:
maps.q[pix_idx]

-0.02841467270396214

In [108]:
maps.u[pix_idx]

-0.002504444878711355

In [24]:
cs.lmax

95

In [5]:
Resolution(128)

Healpix resolution(NSIDE = 128)

In [23]:
test

1×2×3 Array{ComplexF64, 3}:
[:, :, 1] =
 -31.204+9.69749e-13im  -31.204+9.69749e-13im

[:, :, 2] =
 -13.4993-6.74642im  -13.4993-6.74642im

[:, :, 3] =
 -13.4993+6.74642im  -13.4993+6.74642im